In [2]:
import pandas as pd
import numpy as np

In [3]:
dfo = pd.read_csv('dfo_masterlist.csv', encoding='latin1')
emdat = pd.read_csv('historical_dataset.csv', encoding='latin1')

In [8]:
dfo.isnull().sum()

Register__       0
Annual_DFO       0
Glide__       3687
Country__c     326
Other         3730
Nations       3927
Affected      4029
Detailed_L     404
Rivers        2182
Began          326
Ended          326
Duration_i       0
Dead             0
Displaced        0
Damage__US       0
Main_cause     382
Severity__       0
Affected_s       0
Magnitude        0
Centroid_X       0
Centroid_Y       0
"x"_if_act    4018
M_6              0
Total_annu       0
M_4              0
Total_an_1       0
Date_Began     326
Total_floo       0
Total_fl_1       0
F31           4029
F32           4029
F33           4029
F34           4029
F35           4029
F36           4029
dtype: int64

In [9]:
emdat.isnull().sum()

DisNo.                                           0
Historic                                         0
Classification Key                               0
Disaster Group                                   0
Disaster Subgroup                                0
Disaster Type                                    0
Disaster Subtype                                 0
External IDs                                 12522
Event Name                                   11548
ISO                                              0
Country                                          0
Subregion                                        0
Region                                           0
Location                                       798
Origin                                       12585
Associated Types                             13203
OFDA/BHA Response                                0
Appeal                                           0
Declaration                                      0
AID Contribution ('000 US$)    

In [8]:
# cleaning and prepare EM-DAT 

# Filter the dataset to only include Floods
emdat_floods = emdat[emdat['Disaster Type'] == 'Flood'].copy()

# Ensure Year and Month are integers for accurate matching
emdat_floods['Merge_Year'] = emdat_floods['Start Year'].astype(int)
emdat_floods['Merge_Month'] = emdat_floods['Start Month'].fillna(0).astype(int)

# Create a cleaned country column (lowercase, no extra spaces) to ensure perfect matching
emdat_floods['Country_Clean'] = emdat_floods['Country'].astype(str).str.strip().str.lower()

# Select only the columns we need for the ML model
emdat_cols = [
    'Merge_Year', 'Merge_Month', 'Country_Clean', 'Country', 'ISO', 
    'Disaster Subtype', 'Total Deaths', 'No. Affected', 'Total Affected', 
    "Total Damage ('000 US$)"
]

emdat_clean = emdat_floods[emdat_cols].copy()
emdat_clean

,Merge_Year,Merge_Month,Country_Clean,Country,ISO,Disaster Subtype,Total Deaths,No. Affected,Total Affected,Total Damage ('000 US$)
1,2026,3,kenya,Kenya,KEN,Flood (General),59.0,NaN,27.0,NaN
3,2026,4,afghanistan,Afghanistan,AFG,Flood (General),18.0,73300.0,73309.0,NaN
11,2026,5,democratic republic of the congo,Democratic Republic of the Congo,COD,Flood (General),6.0,79000.0,79000.0,NaN
16,2025,8,equatorial guinea,Equatorial Guinea,GNQ,Flood (General),NaN,23115.0,23115.0,NaN
31,2025,3,argentina,Argentina,ARG,Flood (General),125.0,235900.0,236700.0,375000.0
...,...,...,...,...,...,...,...,...,...,...
16900,2019,10,niger,Niger,NER,Flood (General),NaN,23000.0,23000.0,NaN
16901,2019,10,united republic of tanzania,United Republic of Tanzania,TZA,Flood (General),14.0,NaN,NaN,NaN
16902,2019,9,senegal,Senegal,SEN,Flood (General),6.0,8919.0,8968.0,NaN
16911,2019,11,france,France,FRA,Flood (General),5.0,100.0,102.0,NaN


In [9]:
# clean and prepare DFO 

# Drop rows where the start date is completely missing
dfo_clean = dfo.dropna(subset=['Began']).copy()

# Convert the 'Began' string into a standard pandas datetime format
dfo_clean['Began_Date'] = pd.to_datetime(dfo_clean['Began'], errors='coerce')

# Drop any rows where the date conversion failed
dfo_clean = dfo_clean.dropna(subset=['Began_Date']).copy()

# Extract the Year and Month to match with EM-DAT
dfo_clean['Merge_Year'] = dfo_clean['Began_Date'].dt.year.astype(int)
dfo_clean['Merge_Month'] = dfo_clean['Began_Date'].dt.month.astype(int)

# Standardize the Country names just like we did for EM-DAT
dfo_clean['Country_Clean'] = dfo_clean['Country__c'].astype(str).str.strip().str.lower()

# Select the geospatial and severity columns
dfo_cols = [
    'Merge_Year', 'Merge_Month', 'Country_Clean',
    'Centroid_X', 'Centroid_Y', 'Duration_i', 'Dead', 'Displaced', 
    'Severity__', 'Affected_s', 'Magnitude'
]
dfo_selected = dfo_clean[dfo_cols].copy()

dfo_selected

,Merge_Year,Merge_Month,Country_Clean,Centroid_X,Centroid_Y,Duration_i,Dead,Displaced,Severity__,Affected_s,Magnitude
0,2010,8,niger,14.00,13.60,11.0,1.0,5000.0,1.0,619100.0,6.833153
1,2010,8,united states,-93.47,42.31,2.0,1.0,1000.0,1.0,8257.0,4.217852
2,2010,7,north korea,125.71,39.59,13.0,0.0,17000.0,1.0,51190.0,5.823128
3,2010,8,india,76.12,33.53,3.0,150.0,180.0,11.5,145000.0,6.699187
4,2010,7,china,99.58,37.84,16.0,65.0,180.0,1.5,442500.0,7.026125
...,...,...,...,...,...,...,...,...,...,...,...
3698,1985,2,mozambique,0.00,0.00,3.0,19.0,0.0,1.5,20080.0,4.955976
3699,1985,2,indonesia,0.00,0.00,15.0,21.0,300.0,1.0,16540.0,5.394627
3700,1985,1,philippines,0.00,0.00,2.0,43.0,444.0,1.0,12850.0,4.409933
3701,1985,1,brazil,0.00,0.00,19.0,229.0,80000.0,1.5,678500.0,7.286395


In [15]:
# Now we will merge these two datasets on the basis of 
# country , year and month 
# Merge datasets where the Country, Year, and Month are identical
fused_df = pd.merge(
    dfo_selected, 
    emdat_clean, 
    on=['Country_Clean', 'Merge_Year', 'Merge_Month'], 
    how='inner'
)
print(fused_df.shape)
fused_df

(1548, 18)


,Merge_Year,Merge_Month,Country_Clean,Centroid_X,Centroid_Y,Duration_i,Dead,Displaced,Severity__,Affected_s,Magnitude,Country,ISO,Disaster Subtype,Total Deaths,No. Affected,Total Affected,Total Damage ('000 US$)
0,2010,8,niger,14.00,13.60,11.0,1.0,5000.0,1.0,619100.0,6.833153,Niger,NER,Riverine flood,3.0,226611.0,226611.0,NaN
1,2010,8,india,76.12,33.53,3.0,150.0,180.0,11.5,145000.0,6.699187,India,IND,Flash flood,196.0,12500.0,12725.0,NaN
2,2010,7,afghanistan,68.86,35.21,8.0,65.0,180.0,1.0,176800.0,6.150572,Afghanistan,AFG,Riverine flood,65.0,5000.0,5000.0,NaN
3,2010,7,pakistan,73.26,33.36,16.0,1600.0,14000000.0,2.0,129700.0,6.618090,Pakistan,PAK,Flash flood,1985.0,20356550.0,20359496.0,9500000.0
4,2010,7,pakistan,73.26,33.36,16.0,1600.0,14000000.0,2.0,129700.0,6.618090,Pakistan,PAK,Flash flood,60.0,4000.0,4000.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1543,2000,1,philippines,0.00,0.00,5.0,23.0,20000.0,1.0,16700.0,4.921686,Philippines,PHL,Flash flood,50.0,153885.0,153885.0,4080.0
1544,2000,1,philippines,0.00,0.00,5.0,23.0,20000.0,1.0,16700.0,4.921686,Philippines,PHL,Coastal flood,NaN,NaN,5250.0,NaN
1545,2000,1,mozambique,0.00,0.00,62.0,929.0,733000.0,1.5,436000.0,7.607969,Mozambique,MOZ,Riverine flood,800.0,4500000.0,4500000.0,419200.0
1546,2000,1,angola,0.00,0.00,8.0,31.0,70000.0,1.0,47000.0,5.575188,Angola,AGO,Riverine flood,31.0,70000.0,70000.0,10000.0


In [16]:
fused_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1548 entries, 0 to 1547
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Merge_Year               1548 non-null   int64  
 1   Merge_Month              1548 non-null   int64  
 2   Country_Clean            1548 non-null   str    
 3   Centroid_X               1548 non-null   float64
 4   Centroid_Y               1548 non-null   float64
 5   Duration_i               1548 non-null   float64
 6   Dead                     1548 non-null   float64
 7   Displaced                1548 non-null   float64
 8   Severity__               1548 non-null   float64
 9   Affected_s               1548 non-null   float64
 10  Magnitude                1548 non-null   float64
 11  Country                  1548 non-null   str    
 12  ISO                      1548 non-null   str    
 13  Disaster Subtype         1548 non-null   str    
 14  Total Deaths             1299 non-n